<a href="https://colab.research.google.com/github/BurroClurse/BrownBag/blob/main/BrownBagMatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brown Bag Matching Check: Wrong Joins and Wrong Splits

## Setup: Libraries, Java and Node

In [15]:
N_PATIENTS = 300        # how many synthetic patients Synthea makes (300 takes about 3 minutes)
SEED = 42               # same seed, same patients, so runs can be compared
BRANCH = "claude/determined-hypatia-gadqfc"   # use "main" once the reconciliation work is merged

!pip -q install pandas scikit-learn requests
!which java || (apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null)
!node -e "process.exit(Number(process.versions.node.split('.')[0]) >= 20 ? 0 : 1)" || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get -qq install -y nodejs > /dev/null)
!java -version 2>&1 | head -1
!node --version

import json, re, time, random, difflib, glob, os
import requests
import pandas as pd
random.seed(SEED)

/usr/bin/java
[0.007s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
v22.23.4


## Step 1: Get Brown Bag's Matching Code

In [16]:
!rm -rf BrownBag && git clone -q --depth 1 -b {BRANCH} https://github.com/BurroClurse/BrownBag.git
!ls BrownBag/js/reconcile.js BrownBag/js/meds.js

BrownBag/js/meds.js  BrownBag/js/reconcile.js


## Step 2: Make Synthetic Patients with Synthea

In [17]:
!test -f synthea.jar || wget -q -O synthea.jar https://github.com/synthetichealth/synthea/releases/download/master-branch-latest/synthea-with-dependencies.jar
!rm -rf synthea_out && java -jar synthea.jar -p {N_PATIENTS} -s {SEED} --exporter.baseDirectory=./synthea_out > synthea.log 2>&1; tail -2 synthea.log
bundles = [f for f in glob.glob("synthea_out/fhir/*.json") if "Information" not in f]
len(bundles)

120 -- Elisa944 Consuelo468 Wintheiser220 (32 y/o F) Northampton, Massachusetts DECEASED (46506)
122 -- Jeraldine574 Santos184 Rutherford999 (48 y/o F) Ashland, Massachusetts  (67818)


131

## Step 3: Read Every Medicine and Person

In [18]:
RXNORM = "http://www.nlm.nih.gov/research/umls/rxnorm"
people, meds = {}, []
for path in bundles:
    entries = [e["resource"] for e in json.load(open(path))["entry"]]
    patient = next(r for r in entries if r["resourceType"] == "Patient")
    people[patient["id"]] = patient
    seen = set()
    for r in entries:
        if r["resourceType"] != "MedicationRequest" or "medicationCodeableConcept" not in r:
            continue
        coding = next((c for c in r["medicationCodeableConcept"].get("coding", []) if c.get("system") == RXNORM), None)
        if not coding or coding["code"] in seen:
            continue
        seen.add(coding["code"])
        meds.append({"patient": patient["id"], "rxcui": coding["code"], "name": coding.get("display", ""), "resource": r})

med_df = pd.DataFrame([{k: v for k, v in m.items() if k != "resource"} for m in meds])
print(len(people), "patients,", len(med_df), "medicines,", med_df.rxcui.nunique(), "different products")
med_df.head()

KeyboardInterrupt: 

## Step 4: Look Up Each Product in RxNorm

In [ ]:
RXNAV = "https://rxnav.nlm.nih.gov/REST"
cache = json.load(open("rxnorm.json")) if os.path.exists("rxnorm.json") else {}

def get(url):
    for attempt in range(3):
        try:
            time.sleep(0.06)          # RxNav asks for no more than 20 requests a second
            return requests.get(url, timeout=20).json()
        except Exception:
            time.sleep(2)
    return {}

def related(rxcui, ttys):
    groups = get(f"{RXNAV}/rxcui/{rxcui}/related.json?tty={'+'.join(ttys)}").get("relatedGroup", {}).get("conceptGroup", [])
    return {g["tty"]: g.get("conceptProperties", []) for g in groups}

def product(rxcui):
    if rxcui in cache:
        return cache[rxcui]
    props = get(f"{RXNAV}/rxcui/{rxcui}/properties.json").get("properties") or {}
    rel = related(rxcui, ["IN", "PIN", "DF"])
    cache[rxcui] = {
        "name": props.get("name", ""), "tty": props.get("tty", ""),
        "ingredients": sorted(c["rxcui"] for c in rel.get("IN", [])),
        "salts": sorted(c["rxcui"] for c in rel.get("PIN", [])),
        "form": (rel.get("DF") or [{}])[0].get("name", ""),
    }
    return cache[rxcui]

siblings = {}
for code in sorted(med_df.rxcui.unique()):
    p = product(code)
    for ingredient in p["ingredients"][:1]:
        if ingredient not in siblings:
            scds = related(ingredient, ["SCD"]).get("SCD", [])
            siblings[ingredient] = [c["rxcui"] for c in scds][:5]
            for s in siblings[ingredient]:
                product(s)
for code in list(cache):             # every ingredient gets its own entry too
    for ingredient in cache[code]["ingredients"]:
        product(ingredient)
json.dump(cache, open("rxnorm.json", "w"))
print(len(cache), "RxNorm concepts looked up")

## Step 5: Rough the Records Up the Way Real Sources Do

In [ ]:
import copy

def concept(resource):
    return resource["medicationCodeableConcept"]

def ingredient_code_first(r):           # Epic: the ingredient's code listed before the product's
    r = copy.deepcopy(r); p = cache[concept(r)["coding"][0]["code"]]
    if p["ingredients"]:
        concept(r)["coding"].insert(0, {"system": RXNORM, "code": p["ingredients"][0], "display": cache[p["ingredients"][0]]["name"]})
    return r

ABBREVIATIONS = [(" Oral Tablet", " TAB"), (" Oral Capsule", " CAP"), ("Hydrochloride", "HCL"),
                 ("Extended Release", "ER"), (" MG", "MG")]
def caps_and_abbreviations(r):          # pharmacy systems: capitals, TAB, HCL, ER
    r = copy.deepcopy(r); text = concept(r)["coding"][0].get("display", "")
    for long, short in ABBREVIATIONS:
        text = text.replace(long, short)
    concept(r)["text"] = text.upper()
    return r

def text_only(r):                       # no code at all, only words
    r = caps_and_abbreviations(r); concept(r).pop("coding")
    return r

def leading_dot(r):                     # "0.5 MG" written ".5 mg"
    r = text_only(r); concept(r)["text"] = re.sub(r"\b0\.(\d)", r".\1", concept(r)["text"])
    return r

MED_ROUGHERS = [ingredient_code_first, caps_and_abbreviations, text_only, leading_dot]

def last_first_text(p):                 # Cerner: "LAST, FIRST" in name.text only
    p = copy.deepcopy(p); n = p["name"][0]
    p["name"] = [{"text": f"{n['family'].upper()}, {n['given'][0].upper()}"}]
    return p

def no_digits(p):                       # Synthea adds numbers to names; real sources don't
    p = copy.deepcopy(p)
    for n in p["name"]:
        n["family"] = re.sub(r"\d+", "", n.get("family", "")); n["given"] = [re.sub(r"\d+", "", g) for g in n.get("given", [])]
    return p

def datetime_birth_date(p):             # a time stuck on the birth date
    p = copy.deepcopy(p); p["birthDate"] = p["birthDate"] + "T00:00:00Z"
    return p

def nickname_first(p):                  # a nickname listed before the legal name
    p = copy.deepcopy(p); p["name"].insert(0, {"use": "nickname", "given": ["Sunny"]})
    return p

def birth_date_typo(p):                 # day and month swapped: same person, but Brown Bag should keep them apart
    p = copy.deepcopy(p); y, m, d = p["birthDate"].split("-")
    if int(d) <= 12 and d != m:
        p["birthDate"] = f"{y}-{d}-{m}"
    return p

PERSON_ROUGHERS = [last_first_text, no_digits, datetime_birth_date, nickname_first, birth_date_typo]
print(len(MED_ROUGHERS), "medicine and", len(PERSON_ROUGHERS), "person roughers")

## Step 6: Build Test Pairs with Known Answers

In [ ]:
def request_for(rxcui):                 # a plain record of another product, for the pairs that must stay apart
    return {"resourceType": "MedicationRequest", "status": "active", "authoredOn": "2026-01-01",
            "medicationCodeableConcept": {"coding": [{"system": RXNORM, "code": rxcui, "display": cache[rxcui]["name"]}],
                                          "text": cache[rxcui]["name"]}}

def truth(a, b):                        # the right answer, from RxNorm: same ingredients, salts and dose form
    pa, pb = cache[a], cache[b]
    same = pa["ingredients"] == pb["ingredients"] and pa["salts"] == pb["salts"] and pa["form"] == pb["form"]
    return "join" if same else "apart"

pairs = []
for m in meds:
    p = cache.get(m["rxcui"])
    if not p or not p["ingredients"]:
        continue
    for rough in MED_ROUGHERS:          # the same record, roughed up: should join
        pairs.append({"kind": "medicine", "tag": rough.__name__, "truth": "join", "a": m["resource"], "b": rough(m["resource"])})
    for other in siblings.get(p["ingredients"][0], []):
        if other == m["rxcui"] or other not in cache:
            continue
        answer = truth(m["rxcui"], other) # another strength of the same thing joins; another salt or form stays apart
        for rough in [None, ingredient_code_first, text_only]:
            b = request_for(other) if rough is None else rough(request_for(other))
            pairs.append({"kind": "medicine", "tag": "sibling, " + (rough.__name__ if rough else "as written"), "truth": answer, "a": m["resource"], "b": b})

ids = list(people)
for pid in ids:
    for rough in PERSON_ROUGHERS:
        answer = "apart" if rough is birth_date_typo and rough(people[pid])["birthDate"] != people[pid]["birthDate"] else "join"
        pairs.append({"kind": "person", "tag": rough.__name__, "truth": answer, "a": people[pid], "b": rough(people[pid])})
    other = people[random.choice(ids)]
    if other["id"] != pid:              # a different person: must stay apart
        pairs.append({"kind": "person", "tag": "different person", "truth": "apart", "a": people[pid], "b": other})

for i, pair in enumerate(pairs):
    pair["id"] = i
with open("pairs.jsonl", "w") as f:
    f.writelines(json.dumps(p) + "\n" for p in pairs)
pd.DataFrame(pairs).groupby(["kind", "truth"]).size()

## Step 7: Ask Brown Bag's Own Code

In [ ]:
ingredient_map = {code: {"rxcui": "+".join(sorted(p["ingredients"], key=int)), "name": ""}
                  for code, p in cache.items() if p["ingredients"]}
ingredient_map.update({code: {"rxcui": code, "name": p["name"]} for code, p in cache.items() if p["tty"] == "IN"})
json.dump(ingredient_map, open("ingredients.json", "w"))

open("ask_brown_bag.mjs", "w").write("""
import { readFileSync, writeFileSync } from "node:fs";
import { fromMedicationRequest } from "./BrownBag/js/meds.js";
import { matchOf, GROUPS, personOf, samePerson } from "./BrownBag/js/reconcile.js";
const map = new Map(Object.entries(JSON.parse(readFileSync("ingredients.json", "utf8"))));
const lines = readFileSync("pairs.jsonl", "utf8").trim().split("\\n");
const out = lines.map(function (line) {
  const p = JSON.parse(line);
  if (p.kind === "person") return { id: p.id, answer: samePerson(personOf(p.a), personOf(p.b)) ? "join" : "apart", level: null };
  const level = matchOf(fromMedicationRequest(p.a, "A"), fromMedicationRequest(p.b, "B"), map);
  return { id: p.id, answer: GROUPS.indexOf(level) !== -1 ? "join" : "apart", level: level };
});
writeFileSync("brown_bag.jsonl", out.map(function (o) { return JSON.stringify(o); }).join("\\n"));
console.log("Brown Bag answered", out.length, "pairs");
""")
!node ask_brown_bag.mjs

## Step 8: Score Brown Bag's Rules

In [ ]:
answers = pd.DataFrame([json.loads(l) for l in open("brown_bag.jsonl")])
results = pd.DataFrame(pairs)[["id", "kind", "tag", "truth"]].merge(answers, on="id")
results["outcome"] = results.apply(lambda r: "right" if r.truth == r.answer
                                   else ("WRONG JOIN" if r.answer == "join" else "wrong split"), axis=1)
print(results.outcome.value_counts().to_string())
results.pivot_table(index=["kind", "tag"], columns="outcome", values="id", aggfunc="count", fill_value=0)

## Step 9: Turn Each Pair into Numbers for a Model



In [ ]:
def words(resource):
    c = resource.get("medicationCodeableConcept", {})
    return (c.get("text") or (c.get("coding") or [{}])[0].get("display", "")).lower()

def codes(resource):
    return [x["code"] for x in resource.get("medicationCodeableConcept", {}).get("coding", []) if x.get("system") == RXNORM]

STRENGTH = re.compile(r"(\d*\.?\d+)\s*(mg|mcg|ug|g|ml|unt|units?|iu|%)", re.I)
RELEASE = re.compile(r"\b(\d+ hr|er|xr|xl|sr|cr|dr|la|odt|extended|delayed)\b", re.I)
SALTS = re.compile(r"\b(hydrochloride|hcl|tartrate|succinate|sodium|potassium|maleate|besylate|sulfate|citrate)\b", re.I)
FORMS = re.compile(r"\b(tablet|tab|capsule|cap|solution|suspension|injection|patch|cream|ointment|inhaler|spray|drops?)\b", re.I)

def strengths(text):
    return sorted(round(float(v) * (0.001 if u.lower() in ("mcg", "ug") else 1), 6) for v, u in STRENGTH.findall(text))

def features(pair):
    a, b = pair["a"], pair["b"]
    ta, tb = words(a), words(b)
    ca, cb = codes(a), codes(b)
    ia = {i for c in ca for i in cache.get(c, {}).get("ingredients", [])}
    ib = {i for c in cb for i in cache.get(c, {}).get("ingredients", [])}
    first = lambda t: (re.findall(r"[a-z]+", t) or [""])[0]
    return {
        "both_coded": int(bool(ca and cb)),
        "a_code_in_b": int(bool(set(ca) & set(cb))),
        "same_ingredients": int(bool(ia) and ia == ib),
        "same_strengths": int(strengths(ta) == strengths(tb)),
        "same_release": int(sorted(RELEASE.findall(ta)) == sorted(RELEASE.findall(tb))),
        "same_salts": int({s.lower().replace("hcl", "hydrochloride") for s in SALTS.findall(ta)} ==
                          {s.lower().replace("hcl", "hydrochloride") for s in SALTS.findall(tb)}),
        "same_form_word": int({f.lower()[:3] for f in FORMS.findall(ta)} == {f.lower()[:3] for f in FORMS.findall(tb)}),
        "same_first_word": int(first(ta) == first(tb)),
        "name_alike": difflib.SequenceMatcher(None, ta, tb).ratio(),
    }

med_pairs = [p for p in pairs if p["kind"] == "medicine"]
X = pd.DataFrame([features(p) for p in med_pairs])
y = (pd.Series([p["truth"] for p in med_pairs]) == "join").astype(int)
X.describe().T[["mean", "min", "max"]]

## Step 10: Train a Model on the Known Answers

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report

groups = [p["a"].get("subject", {}).get("reference", "") for p in med_pairs]
train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(X, y, groups))
model = GradientBoostingClassifier(random_state=SEED).fit(X.iloc[train], y.iloc[train])
predicted = model.predict(X.iloc[test])

print(confusion_matrix(y.iloc[test], predicted))
print(classification_report(y.iloc[test], predicted, target_names=["apart", "join"]))
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

## Step 11: Find Where the Model and the Rules Disagree

In [ ]:
med_results = results[results.kind == "medicine"].reset_index(drop=True)
med_results["model_join"] = model.predict_proba(X)[:, 1]
sure = (med_results.model_join > 0.95) | (med_results.model_join < 0.05)
model_says = med_results.model_join.round().map({1: "join", 0: "apart"})
flagged = med_results[sure & (model_says != med_results.answer)].copy()
flagged["a_name"] = [words(med_pairs[i]["a"]) for i in flagged.index]
flagged["b_name"] = [words(med_pairs[i]["b"]) for i in flagged.index]
print(len(flagged), "pairs to review, in these groups:")
print(flagged.groupby(["tag", "truth", "answer"]).size().sort_values(ascending=False).to_string())
# Three examples from each group: one group is usually one gap in the rules
flagged.groupby(["tag", "truth", "answer"]).head(3)[["tag", "truth", "answer", "level", "model_join", "a_name", "b_name"]]

## Step 12: Try It on Real Product Names from openFDA

In [ ]:
INGREDIENTS = ["metformin", "metoprolol", "lisinopril", "atorvastatin", "bupropion", "diltiazem", "levothyroxine", "methotrexate"]
products = []
for name in INGREDIENTS:
    data = get(f'https://api.fda.gov/drug/ndc.json?search=generic_name:"{name}"&limit=100').get("results", [])
    for d in data:
        strength = " / ".join(f"{i['name']} {i['strength']}" for i in d.get("active_ingredients", []))
        products.append({"ingredient": name, "text": f"{d.get('generic_name', '')} {strength} {d.get('dosage_form', '')}",
                         "rxcuis": d.get("openfda", {}).get("rxcui", [])})

def as_record(text):
    return {"resourceType": "MedicationRequest", "status": "active", "authoredOn": "2026-01-01",
            "medicationCodeableConcept": {"text": text}}

real_pairs = []
for name in INGREDIENTS:
    group = [p for p in products if p["ingredient"] == name][:30]
    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            if group[i]["text"] != group[j]["text"]:
                real_pairs.append({"a": as_record(group[i]["text"]), "b": as_record(group[j]["text"]),
                                   "fda_says": "join" if set(group[i]["rxcuis"]) & set(group[j]["rxcuis"]) else "apart"})
RX = pd.DataFrame([features(p) for p in real_pairs])
real = pd.DataFrame({"a": [p["a"]["medicationCodeableConcept"]["text"] for p in real_pairs],
                     "b": [p["b"]["medicationCodeableConcept"]["text"] for p in real_pairs],
                     "fda_says": [p["fda_says"] for p in real_pairs],
                     "model_join": model.predict_proba(RX)[:, 1]})
print(len(products), "real products,", len(real_pairs), "pairs")
real[(real.model_join > 0.95) & (real.fda_says == "apart")].head(15)

## Step 13: Save the Flagged Cases for Brown Bag's Tests

In [ ]:
cases = [{"case": f"generated {int(r.id)}", "tag": r.tag, "truth": r.truth, "brown_bag": r.answer,
          "level": r.level, "model_join": round(float(r.model_join), 3),
          "a": med_pairs[i]["a"]["medicationCodeableConcept"], "b": med_pairs[i]["b"]["medicationCodeableConcept"]}
         for i, r in flagged.iterrows()]
json.dump({"made": time.strftime("%Y-%m-%d"), "patients": N_PATIENTS, "seed": SEED,
           "wrong_joins": int((results.outcome == "WRONG JOIN").sum()),
           "wrong_splits": int((results.outcome == "wrong split").sum()),
           "cases": cases}, open("brown-bag-generated-cases.json", "w"), indent=1)
print(len(cases), "cases saved")
try:
    from google.colab import files
    files.download("brown-bag-generated-cases.json")
except ImportError:
    print("Not in Colab: the file is brown-bag-generated-cases.json")